# 3-4 Pandas文件读写

IO 是 Input / Output 的缩写，表示数据的输入与输出。Pandas 可以读取 CSV、Excel、JSON、HTML 等文件，并将结果写出。

> 约定：使用 `pd` 作为 Pandas 的别名，使用 `np` 作为 NumPy 的别名。本节可独立从上到下运行。

In [ ]:
import pandas as pd
import numpy as np

## 1. 准备数据路径

本节读取本章 `data` 文件夹中的模拟练习数据，使用 `pathlib` 处理相对路径。

In [ ]:
from pathlib import Path

# 从本章目录运行时使用 data；从项目根目录运行时定位到本章。
data_dir = Path("data")
if not data_dir.is_dir():
    data_dir = Path("第3章-Pandas") / "data"
if not data_dir.is_dir():
    raise FileNotFoundError("未找到本章的 data 文件夹。")

data_dir.as_posix()

## 2. 常见的读写方法

| 格式 | 读取 | 写出 |
|---|---|---|
| CSV / 分隔文本 | `pd.read_csv()` | `df.to_csv()` |
| Excel | `pd.read_excel()` | `df.to_excel()` |
| JSON | `pd.read_json()` | `df.to_json()` |
| HTML 表格 | `pd.read_html()` | `df.to_html()` |
| Python 字典 | `pd.DataFrame()` | `df.to_dict()` |

Excel 示例需要 `openpyxl`，HTML 示例需要 `lxml`。如环境缺少这两个库，可在环境中安装后重新运行。

### 2.1 查看本地文件

`Path.iterdir()` 可以跨平台列出文件，不需要调用终端命令。

In [ ]:
sorted(p.name for p in data_dir.iterdir() if p.is_file())

## 3. 读取文本与 Excel

### 3.1 读取 CSV

默认第一行为列名、逗号为分隔符；默认重新生成从 0 开始的行索引。

In [ ]:
df_csv = pd.read_csv(data_dir / "df.csv", encoding="utf-8")
df_csv

### 3.2 读取 TXT

扩展名不决定分隔符。这里的 `df.txt` 也是逗号分隔，所以仍可用 `read_csv()`；制表符分隔文件则设置 `sep="\t"`。

In [ ]:
df_txt = pd.read_csv(data_dir / "df.txt", sep=",", encoding="utf-8")
df_txt

### 3.3 读取 Excel

`sheet_name=0` 表示第一个工作表。

In [ ]:
df_excel = pd.read_excel(data_dir / "df.xlsx", sheet_name=0, engine="openpyxl")
df_excel

## 4. 读取 JSON 与本地 HTML

这两个示例读取 `data` 文件夹中的 JSON 和 HTML 文件，不需要联网。`read_html()` 返回的是 DataFrame **列表**，因为一个页面中可能存在多个表格。

In [ ]:
df_json = pd.read_json(data_dir / "df.json")
df_json

In [ ]:
tables = pd.read_html(data_dir / "df3.html", index_col=0, encoding="utf-8")
len(tables), tables[0].shape

In [ ]:
tables[0]

本地 HTML 带有表格的行索引，所以通过 `index_col=0` 将第一列识别为索引。

`pd.read_clipboard()` 需要系统剪贴板支持，且结果依赖当时复制的内容。可在本机手动复制表格后调用；为了让本节每次运行一致，不把它放入自动执行单元格。

## 5. 写出数据

所有输出保存在笔记章节下的 `输出示例` 文件夹，与 `data` 中的输入文件分开保存。`index=False` 表示不额外写出行索引；`header=True` 表示保留列名。重新运行会更新这些示例输出。

In [ ]:
output_dir = data_dir.parent / "输出示例"
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

In [ ]:
df_excel.to_csv(output_dir / "人员表.csv", index=False, encoding="utf-8-sig")
df_excel.to_excel(output_dir / "人员表.xlsx", index=False, header=True, engine="openpyxl")
df_excel.to_html(output_dir / "人员表.html", index=False, encoding="utf-8")
df_excel.to_json(output_dir / "人员表.json", orient="records", force_ascii=False, indent=2)

`utf-8-sig` 便于 Excel 识别 CSV 的中文编码。JSON 的 `orient="records"` 表示每一行保存为一个字典。

In [ ]:
df_excel.to_dict(orient="records")

`to_dict()` 返回 Python 对象，本身不会生成磁盘文件。默认 `orient="dict"` 则返回“列名 → 索引和值”的嵌套字典。

In [ ]:
df_excel.to_dict()

## 6. 重新读取，检查输出

重新读回 CSV，确认没有多出无名索引列。

In [ ]:
round_trip = pd.read_csv(output_dir / "人员表.csv", encoding="utf-8-sig")
round_trip

In [ ]:
round_trip.equals(df_excel)

结果应为 `True`，说明本例的数据和类型保持一致。不同格式可能改变日期或缺失值类型，实际工作中还应检查 `dtypes`。

## 7. 要点总结

- `read_*` 读取文件，`to_*` 写出文件或转换数据。
- 文本读取应关注分隔符、编码、表头和索引，而不只看扩展名。
- `read_html()` 返回表格列表；剪贴板读取需要本机交互环境。
- 写出时明确 `index`，把输出与原始数据分开保存。